# Lab 0 Foundations: Train a Tiny LLM on a Book Excerpt

This notebook trains a tiny word-level transformer on a short public-domain excerpt from *The Wonderful Wizard of Oz*.

The goal is not to build a production LLM. The goal is to let you see a small training loop, watch loss improve, and inspect how a trained model predicts likely next words from context.


## Step 1: Setup

This notebook uses only local files and PyTorch. It does not require Ollama or `.env`.

If the next cell fails on the `torch` import, install the base course dependencies from the repo root first:

```bash
pip install -r requirements.txt
```


In [ ]:
import random
import re
import time
from pathlib import Path

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'PyTorch is required for this notebook. Run `pip install -r requirements.txt` from the repo root and try again.'
    ) from exc

LAB_NAME = 'lab0_0_llm_foundations'
lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

data_dir = lab_dir / 'data'
corpus_path = data_dir / 'wizard_of_oz_excerpt.txt'
if not corpus_path.exists():
    raise FileNotFoundError('Could not find wizard_of_oz_excerpt.txt in data/.')

random.seed(7)
torch.manual_seed(7)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Lab folder:', lab_dir)
print('Corpus path:', corpus_path)
print('Torch version:', torch.__version__)
print('Device:', device)


## Step 2: Load the Book Excerpt

The corpus is a short public-domain excerpt from *The Wonderful Wizard of Oz*.

To keep the training visible on classroom hardware, the notebook will repeat this excerpt several times. That repeated corpus is still only a teaching dataset, not a realistic training corpus.


In [ ]:
raw_text = corpus_path.read_text(encoding='utf-8').strip()
print(raw_text[:1800])
print('\n---')
print('Original excerpt character count:', len(raw_text))
print('Original excerpt word count:', len(raw_text.split()))


## Step 3: Tokenize at the Word Level

This notebook uses a simple word-level tokenizer with punctuation tokens.

Real LLMs often use more advanced subword tokenization. Here we keep the tokenizer simple so students can focus on the training and prediction steps.


In [ ]:
repeat_factor = 6
training_text = '\n\n'.join([raw_text] * repeat_factor)

token_pattern = re.compile(r"[A-Za-z']+|[.,;:!?\-]")

def tokenize(text: str) -> list[str]:
    return token_pattern.findall(text.lower())

def detokenize(tokens: list[str]) -> str:
    pieces = []
    punctuation = {'.', ',', ';', ':', '!', '?'}
    for token in tokens:
        if token in punctuation and pieces:
            pieces[-1] += token
        elif token == '-' and pieces:
            pieces[-1] += token
        else:
            pieces.append(token)
    return ' '.join(pieces)

tokens = tokenize(training_text)
vocab = sorted(set(tokens))
stoi = {token: index for index, token in enumerate(vocab)}
itos = {index: token for token, index in stoi.items()}

def encode(token_list: list[str]) -> list[int]:
    return [stoi[token] for token in token_list]

def decode(index_list: list[int]) -> list[str]:
    return [itos[index] for index in index_list]

ids = torch.tensor(encode(tokens), dtype=torch.long)

print('Repeat factor:', repeat_factor)
print('Token count:', len(tokens))
print('Vocabulary size:', len(vocab))
print('First 40 tokens:')
print(tokens[:40])


## Step 4: Build Training and Validation Splits

The model will see short windows of context and learn to predict the next word at each position.


In [ ]:
split_index = int(0.9 * len(ids))
train_ids = ids[:split_index]
val_ids = ids[split_index:]

block_size = 12
batch_size = 32

def get_batch(split: str):
    data = train_ids if split == 'train' else val_ids
    starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[start:start + block_size] for start in starts])
    y = torch.stack([data[start + 1:start + block_size + 1] for start in starts])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('Train token count:', len(train_ids))
print('Validation token count:', len(val_ids))
print('One batch input shape:', tuple(xb.shape))
print('One batch target shape:', tuple(yb.shape))
print('Example input tokens:')
print(detokenize(decode(xb[0].tolist())))
print('\nExpected next-token targets:')
print(decode(yb[0].tolist()))


## Step 5: Define a Tiny Word-Level Transformer

This is a compact causal transformer. It is still much smaller than a real production LLM, but it uses the same basic training idea: read prior context and predict the next token.


In [ ]:
class Head(nn.Module):
    def __init__(self, head_size: int):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size_local, time_steps, channels = x.shape
        k = self.key(x)
        q = self.query(x)
        weights = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        weights = weights.masked_fill(self.tril[:time_steps, :time_steps] == 0, float('-inf'))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)
        v = self.value(x)
        return weights @ v


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads: int, head_size: int):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    def __init__(self, embedding_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int):
        super().__init__()
        head_size = embedding_dim // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size)
        self.ffwd = FeedForward(embedding_dim)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class TinyWordTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        batch_size_local, time_steps = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        position_embeddings = self.position_embedding_table(torch.arange(time_steps, device=idx.device))
        x = token_embeddings + position_embeddings
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            batch_size_local, time_steps, channels = logits.shape
            logits = logits.view(batch_size_local * time_steps, channels)
            targets = targets.view(batch_size_local * time_steps)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits.view(idx_cond.shape[0], idx_cond.shape[1], vocab_size)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


## Step 6: Train the Model

This training loop is intentionally small. The goal is to make the learning process visible, not to optimize for scale.

Because the excerpt is repeated, the model should learn recurring local patterns quickly enough for students to observe falling loss and better next-word predictions.


In [ ]:
vocab_size = len(vocab)
n_embd = 64
n_head = 4
n_layer = 2
dropout = 0.1
learning_rate = 3e-3
eval_interval = 50
eval_iters = 20
max_iters = 300 if device.type == 'cpu' else 500

model = TinyWordTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

def count_parameters(module: nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters())

@torch.no_grad()
def estimate_loss():
    losses = {}
    model.eval()
    for split in ['train', 'val']:
        split_losses = torch.zeros(eval_iters)
        for iteration in range(eval_iters):
            xb_local, yb_local = get_batch(split)
            _, loss_local = model(xb_local, yb_local)
            split_losses[iteration] = loss_local.item()
        losses[split] = split_losses.mean().item()
    model.train()
    return losses

@torch.no_grad()
def estimate_accuracy(split: str, eval_steps: int = 20):
    correct = 0
    total = 0
    model.eval()
    for _ in range(eval_steps):
        xb_local, yb_local = get_batch(split)
        logits_local, _ = model(xb_local)
        logits_local = logits_local.view(xb_local.shape[0], xb_local.shape[1], vocab_size)
        predictions = logits_local.argmax(dim=-1)
        correct += (predictions == yb_local).sum().item()
        total += yb_local.numel()
    model.train()
    return correct / total

history = []
start_time = time.perf_counter()

print('Parameter count:', count_parameters(model))
print('Starting training...')

for step in range(max_iters + 1):
    if step % eval_interval == 0:
        losses = estimate_loss()
        train_acc = estimate_accuracy('train')
        val_acc = estimate_accuracy('val')
        history.append(
            {
                'step': step,
                'train_loss': losses['train'],
                'val_loss': losses['val'],
                'train_accuracy': train_acc,
                'val_accuracy': val_acc,
            }
        )
        print(
            f"step {step:>3} | train loss {losses['train']:.3f} | val loss {losses['val']:.3f} | "
            f"train acc {train_acc:.3f} | val acc {val_acc:.3f}"
        )

    xb_local, yb_local = get_batch('train')
    _, loss_local = model(xb_local, yb_local)
    optimizer.zero_grad(set_to_none=True)
    loss_local.backward()
    optimizer.step()

elapsed = time.perf_counter() - start_time
print(f'Finished training in {elapsed:.1f} seconds.')


## Step 7: Review Loss and Accuracy

Loss shows how wrong the model still is on average. Accuracy here is the fraction of token positions where the top predicted next word matches the target word.

Because this is a tiny model trained on a repeated small excerpt, these numbers are only teaching signals, not a serious benchmark.


In [ ]:
for row in history:
    print(
        f"step {row['step']:>3}: train loss={row['train_loss']:.3f}, val loss={row['val_loss']:.3f}, "
        f"train acc={row['train_accuracy']:.3f}, val acc={row['val_accuracy']:.3f}"
    )

final_val_loss = history[-1]['val_loss']
if final_val_loss < 20:
    print('\nApproximate validation perplexity:', round(float(torch.exp(torch.tensor(final_val_loss))), 2))


## Step 8: Inspect Next-Word Predictions

Now use the trained model at inference time. The weights are fixed. The model reads the prompt context and scores likely next words.


In [ ]:
@torch.no_grad()
def top_next_words(prompt: str, top_k: int = 5):
    prompt_tokens = tokenize(prompt)
    if not prompt_tokens:
        raise ValueError('Prompt must contain at least one token.')
    prompt_ids = encode(prompt_tokens[-block_size:])
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    logits_local, _ = model(idx)
    logits_local = logits_local.view(1, len(prompt_ids), vocab_size)
    probs = F.softmax(logits_local[0, -1], dim=-1)
    values, indices = torch.topk(probs, k=min(top_k, vocab_size))
    return [(itos[index.item()], values[position].item()) for position, index in enumerate(indices)]

prompts = [
    "there's a cyclone",
    'dorothy soon closed her eyes and fell fast',
    'toto was not',
]

for prompt in prompts:
    print(f'Prompt: {prompt!r}')
    for token, probability in top_next_words(prompt):
        print(f'  {token:<12} {probability:.3f}')
    print()


## Step 9: Generate a Short Continuation

Generation repeats the same loop many times: predict a next token, append it, and continue.


In [ ]:
prompt = 'dorothy stood in the door with toto'
prompt_ids = torch.tensor([encode(tokenize(prompt))], dtype=torch.long, device=device)
generated_ids = model.generate(prompt_ids, max_new_tokens=30, temperature=0.8)[0].tolist()
generated_text = detokenize(decode(generated_ids))

print('Prompt:\n')
print(prompt)
print('\nGenerated continuation:\n')
print(generated_text)


## Reflection

Use these questions to connect this notebook back to the Lab 0 reading:

1. What changed during training, and what stayed fixed during inference?
2. Why does the model predict different next words for different prompt contexts?
3. Why does a tiny repeated corpus help this small model learn visible patterns quickly?
4. Why is this still only a teaching model rather than a production LLM?
5. What limitation in this notebook helps explain why later labs add prompt rules, tools, and human review?
